# Circular Imports and How to Fix Them

One of the most confusing errors you'll encounter:

```
ImportError: cannot import name 'SomeClass' from 'module'
```

The code looks correct. The class exists. Why doesn't it work?

Often, the answer is **circular imports**.

## What Is a Circular Import?

When two modules try to import each other:

```python
# module_a.py
from module_b import ClassB

class ClassA:
    def use_b(self):
        return ClassB()

# module_b.py  
from module_a import ClassA

class ClassB:
    def use_a(self):
        return ClassA()
```

**The problem:**
1. Python starts loading `module_a`
2. It sees `from module_b import ClassB`
3. Python starts loading `module_b`
4. It sees `from module_a import ClassA`
5. But `module_a` isn't finished loading! `ClassA` doesn't exist yet!
6. **ImportError**

## Understanding Import-Time Execution

**Key insight:** Python runs code when you import it.

This is different from many other languages!

In [ ]:
# When Python imports a module, ALL top-level code runs:

# Imagine this file: example_module.py
example_code = '''
print("1. Module is being imported!")  # This runs at import time!

MY_CONSTANT = "hello"  # This runs at import time!
print(f"2. Set constant to: {MY_CONSTANT}")

class MyClass:  # Class definition runs at import time!
    print("3. Class body is executing!")
    
    def method(self):  # But method body only runs when called
        print("4. Method called")

print("5. Module finished loading")
'''

print("What runs when you import this module?")
print(example_code)

In [ ]:
# Let's see import-time execution in action
# We'll create a temporary module and import it

import sys
import types

# Create a module that prints when imported
demo_module = types.ModuleType('demo_import')
demo_code = '''
print(">>> Module code is running at import time!")
x = 42
print(f">>> x has been set to {x}")
'''
exec(demo_code, demo_module.__dict__)

print("\nThe code above ran just from being loaded!")
print(f"We can now access demo_module.x = {demo_module.x}")

## Why Circular Imports Happen

Circular imports often appear when:

1. **Related classes reference each other**
```python
# user.py
from order import Order
class User:
    orders: list[Order]

# order.py  
from user import User
class Order:
    customer: User
```

2. **Type hints reference other modules**
```python
# Same problem, even just for type hints!
from other_module import SomeClass

def process(item: SomeClass) -> None:
    ...
```

3. **Utilities import models and vice versa**

## Solutions

### Solution 1: Move Import Inside Function

Import when you need it, not at module load time.

In [ ]:
# Instead of:
# from module_b import ClassB  # At top level
# 
# class ClassA:
#     def use_b(self):
#         return ClassB()

# Do this:
class ClassA:
    def use_b(self):
        from collections import Counter  # Import inside function
        return Counter(["a", "b", "a"])

# The import only happens when use_b() is called,
# by which time all modules are fully loaded

obj = ClassA()
print(obj.use_b())

### Solution 2: Use `TYPE_CHECKING` for Type Hints

If you only need the import for type hints:

In [ ]:
from typing import TYPE_CHECKING

# This block only runs when type checkers (like mypy) analyze the code
# It does NOT run at runtime!
if TYPE_CHECKING:
    from collections import OrderedDict  # Only for type hints

class MyClass:
    # Use string annotation ("OrderedDict") instead of actual type
    def process(self, data: "OrderedDict") -> None:
        print(f"Processing data: {data}")

# This works because:
# 1. The import doesn't happen at runtime
# 2. The type hint is a string, not evaluated at runtime
# 3. Type checkers see the import and understand the type

obj = MyClass()
obj.process({"a": 1, "b": 2})

### Solution 3: Use String Annotations (Forward References)

Put the type in quotes to delay evaluation:

In [ ]:
# Without quotes - Python evaluates immediately
# class User:
#     friend: User  # Error! User isn't fully defined yet!

# With quotes - Python treats it as a string (forward reference)
class User:
    name: str
    friend: "User"  # Works! Evaluated later
    
    def __init__(self, name: str):
        self.name = name
        self.friend = None

alice = User("Alice")
bob = User("Bob")
alice.friend = bob
print(f"{alice.name}'s friend is {alice.friend.name}")

### Solution 4: Restructure Your Code

Often the best solution is to reorganize:

```python
# Before: circular dependency
# user.py imports order.py
# order.py imports user.py

# After: shared module
# base.py - shared base classes, no imports from user/order
# user.py imports base.py
# order.py imports base.py
```

Or move the problematic code to a third module.

## Common Error Messages

Learn to recognize circular import errors:

In [ ]:
# Error 1: "cannot import name 'X' from 'module'"
# This often indicates circular import - X exists but isn't loaded yet

error_1 = """
ImportError: cannot import name 'UserModel' from 'models.user'

This usually means:
- models.user exists
- UserModel is defined in it
- BUT there's a circular import preventing it from loading
"""
print(error_1)

In [ ]:
# Error 2: "partially initialized module 'X' has no attribute 'Y'"
# Explicit circular import error in Python 3.x

error_2 = """
AttributeError: partially initialized module 'mymodule' has no attribute 'MyClass' 
(most likely due to a circular import)

This is Python telling you directly: circular import!
"""
print(error_2)

## Debugging Circular Imports

Steps to find and fix circular imports:

1. **Find the cycle**
   - Look at the error traceback
   - Trace imports: A imports B imports C imports A

2. **Identify the dependency**
   - Why does A need B?
   - Is it for type hints? Runtime code? Both?

3. **Apply the right fix**
   - Type hints only → `TYPE_CHECKING`
   - Runtime but not at import → move import inside function
   - Fundamental design issue → restructure

In [ ]:
# Debugging tip: Add print statements to see import order

debug_code = '''
# At the top of each file, add:
print(f"Loading {__name__}")

# After each import, add:
from other_module import Something
print(f"{__name__}: imported Something from other_module")

# At the end of the file, add:
print(f"{__name__}: finished loading")
'''

print("Add these debug statements to trace your imports:")
print(debug_code)

## AI Code Pattern: Defensive Imports

AI often generates this pattern to avoid issues:

In [ ]:
# Common AI-generated pattern
from __future__ import annotations  # Makes ALL annotations strings
from typing import TYPE_CHECKING, Optional

if TYPE_CHECKING:
    from collections import Counter  # Only for type checking

class DataProcessor:
    """Process data with optional Counter support."""
    
    def count_items(self, items: list) -> "Counter":
        """Return a Counter of items."""
        from collections import Counter  # Import at runtime when needed
        return Counter(items)

# Test it
processor = DataProcessor()
result = processor.count_items(["a", "b", "a", "c", "b", "a"])
print(f"Counts: {result}")

## Summary

| Problem | Solution |
|---------|----------|
| Circular import for runtime use | Move import inside function |
| Circular import for type hints | Use `TYPE_CHECKING` block |
| Self-referencing type hints | Use string annotation `"ClassName"` |
| Complex circular dependencies | Restructure code, create base module |

### Key Takeaways

1. Python runs code at import time
2. Circular imports fail because modules aren't fully loaded
3. The fix depends on *why* you need the import
4. `TYPE_CHECKING` is your friend for type hints
5. When in doubt, restructure the code

---

## Module Complete!

You now understand Python's import system:

- All the different import forms
- Standard library vs. third-party packages
- How packages and modules are organized
- Why circular imports happen and how to fix them

This knowledge will help you debug the most common errors in AI-generated code!